# Balance Checks Between Conditions

Do the control and treatment conditions differ in demographics or control variables, in the
onboarding sample (N = 74) and the offboarding sample (N = 56)?

In [ ]:
import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats

DATA_DIR = '../data/'
N_PERMUTATIONS, SEED = 10_000, 20261112   # fixed seed: reproducible permutation p-values

# Education in three levels: sd_edu_be_rec code -> level, as in Table 1.
EDU_LEVELS = {1: 1, 2: 1, 3: 1, 4: 2, 5: 2, 6: 3, 7: 3}

CONTINUOUS = {
    'sd_age_years': 'Age',
    'news_interest_': 'News interest',
    'interest_politics_': 'Political interest',
    'ideo_lr_1': 'Left-right ideology',
    'div_values_mean': 'Diversity-seeking values',
    'div_traits_mean': 'Diversity-seeking traits',
    'div_perceived_mean': 'Perceived diversity of news media',
    'inst_trust_1': 'Trust in social media',
    'inst_trust_2': 'Trust in news media',
    'internet_use': 'Internet use',
    'news_use_6': 'News use on social media',
}
CATEGORICAL = {'sd_gender': 'Gender', 'edu_level': 'Education'}

## Load data

In [ ]:
def read_sample(path):
    """One row per participant, with education levels added.

    The data have one row per participant-day; survey answers repeat on every row.
    """
    df = pd.read_csv(path, sep=';', decimal=',', encoding='utf-8-sig').drop_duplicates('handle')
    df['edu_level'] = df['sd_edu_be_rec'].map(EDU_LEVELS)
    return df


samples = {
    'Onboarding and active': read_sample(DATA_DIR + 'onboarding_sample_round1.csv'),
    'Also offboarding': read_sample(DATA_DIR + 'offboarding_sample_round1.csv'),
}
for name, df in samples.items():
    print(name, df.shape)

## Tests

In [ ]:
def chi2_permutation(arm, values):
    """Chi-square statistic of a condition-by-category table, with a permutation p-value.

    Gender and education have expected cell counts below 5, where the chi-square distribution
    is a poor approximation; shuffling the condition labels gives the p-value without it.
    """
    arm_codes = pd.factorize(arm)[0]
    value_codes, levels = pd.factorize(values)

    def chi2(codes):
        table = np.bincount(codes * len(levels) + value_codes, minlength=2 * len(levels))
        return stats.chi2_contingency(table.reshape(2, len(levels)), correction=False)[0]

    observed = chi2(arm_codes)
    rng = np.random.default_rng(SEED)
    permuted = np.array([chi2(rng.permutation(arm_codes)) for _ in range(N_PERMUTATIONS)])
    return observed, len(levels) - 1, (permuted >= observed - 1e-9).mean()


def balance(df):
    """Welch's t-test per continuous variable, chi-square test per categorical variable."""
    control, treatment = df[df['arm'] == 'K'], df[df['arm'] == 'M']
    rows = []
    for var, label in CONTINUOUS.items():
        test = pg.ttest(treatment[var].dropna(), control[var].dropna(), correction=True).iloc[0]
        rows.append({'variable': label, 'test': 'Welch t', 'statistic': test['T'], 'df': test['dof'],
                     'p': test['p-val'], 'effect size (d or V)': test['cohen-d']})
    for var, label in CATEGORICAL.items():
        data = df[['arm', var]].dropna()
        chi2, dof, p = chi2_permutation(data['arm'], data[var])
        rows.append({'variable': label, 'test': 'chi-square, permutation p', 'statistic': chi2, 'df': dof,
                     'p': p, 'effect size (d or V)': np.sqrt(chi2 / len(data))})   # Cramér's V, 2 conditions
    return pd.DataFrame(rows).set_index('variable').round(3)


for name, df in samples.items():
    print(f'{name} (N = {len(df)})')
    display(balance(df))